## 1. Settings and corpus

Byte-Level BPE starts with 256 byte tokens. A target of 270 allows up to 13 learned tokens after the special token.

In [1]:
from collections import Counter
from pathlib import Path
import sys

corpus = ['hello hello world', 'café 👋', 'नमस्ते दुनिया']
TARGET_SIZE = 270
print(sys.executable)

d:\project\.venv\Scripts\python.exe


## 2. Reversible byte mapping

Map every byte to a printable Unicode symbol. This lets BPE operate on strings while retaining every possible byte.

In [2]:
def bytes_to_unicode():
    bs = list(range(ord('!'), ord('~') + 1)) + list(range(ord('¡'), ord('¬') + 1)) + list(range(ord('®'), ord('ÿ') + 1))
    cs = bs.copy()
    extra = 0
    for value in range(256):
        if value not in bs:
            bs.append(value); cs.append(256 + extra); extra += 1
    return dict(zip(bs, map(chr, cs)))

byte_encoder = bytes_to_unicode()
byte_decoder = {symbol: value for value, symbol in byte_encoder.items()}
assert len(byte_encoder) == 256
sample = 'café 👋'
mapped = ''.join(byte_encoder[b] for b in sample.encode('utf-8'))
restored = bytes(byte_decoder[c] for c in mapped).decode('utf-8')
print(mapped, '->', restored)

cafÃ©ĠðŁĳĭ -> café 👋


## 3. Pre-tokenize text

The GPT-2 pattern keeps leading spaces with many chunks, separates contractions, and supports Unicode letters and numbers.

In [3]:
import regex
pattern = regex.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+")
print(pattern.findall("Hello, café! I'm here."))

['Hello', ',', ' café', '!', ' I', "'m", ' here', '.']


## 4. Build the training chunks

Convert each pre-tokenized chunk to UTF-8 bytes, then mapped symbols. Frequencies make repeated chunks more important.

In [4]:
chunks = Counter()
for text in corpus:
    for chunk in pattern.findall(text):
        symbols = tuple(byte_encoder[b] for b in chunk.encode('utf-8'))
        chunks[symbols] += 1
print('Unique chunks:', len(chunks))
print(chunks.most_common(3))

Unique chunks: 15
[(('h', 'e', 'l', 'l', 'o'), 1), (('Ġ', 'h', 'e', 'l', 'l', 'o'), 1), (('Ġ', 'w', 'o', 'r', 'l', 'd'), 1)]


## 5. Count and merge adjacent pairs

BPE selects the most frequent pair. Lexical order breaks frequency ties so runs are reproducible.

In [5]:
def merge(symbols, pair):
    result=[]; i=0
    while i < len(symbols):
        if i+1 < len(symbols) and symbols[i:i+2] == pair:
            result.append(pair[0]+pair[1]); i += 2
        else:
            result.append(symbols[i]); i += 1
    return tuple(result)

pair_counts = Counter()
for symbols, frequency in chunks.items():
    for pair in zip(symbols, symbols[1:]): pair_counts[pair] += frequency
best = min(pair_counts, key=lambda pair: (-pair_counts[pair], pair))
print('First merge:', best, 'count:', pair_counts[best])

First merge: ('à', '¤') count: 9


## 6. Train merge rules

Repeat counting and merging. A tiny corpus may run out of pairs before reaching the requested size.

In [6]:
training_chunks = chunks.copy(); merges=[]; known=set(byte_encoder.values()) | {'<|endoftext|>'}
while len(known) < TARGET_SIZE:
    counts=Counter()
    for symbols, frequency in training_chunks.items():
        for pair in zip(symbols, symbols[1:]): counts[pair] += frequency
    if not counts: break
    pair=min(counts, key=lambda item: (-counts[item], item))
    training_chunks=Counter({merge(symbols,pair): frequency for symbols,frequency in training_chunks.items()})
    merges.append(pair); known.add(pair[0]+pair[1])
print('Learned merges:', merges)

Learned merges: [('à', '¤'), ('à', '¥'), ('e', 'l'), ('el', 'l'), ('ell', 'o'), ('h', 'ello'), ('à¤', '¨'), ('a', 'f'), ('af', 'Ã'), ('afÃ', '©'), ('c', 'afÃ©'), ('l', 'd'), ('o', 'r')]


## 7. Encode by merge rank

Start from bytes again. Apply the earliest learned adjacent pair until no learned pair remains.

In [7]:
ranks={pair: rank for rank,pair in enumerate(merges)}
def encode_tokens(text):
    output=[]
    for chunk in pattern.findall(text):
        symbols=tuple(byte_encoder[b] for b in chunk.encode('utf-8'))
        while len(symbols)>1:
            candidates={p for p in zip(symbols,symbols[1:]) if p in ranks}
            if not candidates: break
            symbols=merge(symbols,min(candidates,key=ranks.__getitem__))
        output.extend(symbols)
    return output

text='unseen 🧪 café नमस्ते'
tokens=encode_tokens(text)
print(tokens)

['u', 'n', 's', 'e', 'e', 'n', 'Ġ', 'ð', 'Ł', '§', 'ª', 'Ġ', 'cafÃ©', 'Ġ', 'à¤¨', 'à¤', '®', 'à¤', '¸', 'à¥', 'į', 'à¤', '¤', 'à¥', 'ĩ']


## 8. Decode and compare with the project implementation

Split merged tokens back into mapped characters, recover bytes, and decode UTF-8. Because all 256 bytes exist, unseen Unicode text still round-trips.

In [8]:
decoded=bytes(byte_decoder[c] for token in tokens for c in token).decode('utf-8')
assert decoded == text
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'ByteLevelBPE'/'code').is_dir())
sys.path.insert(0,str(root))
from ByteLevelBPE.code import ByteLevelBPETokenizer
project_tokenizer=ByteLevelBPETokenizer(merges)
ids=project_tokenizer.encode(text)
assert project_tokenizer.decode(ids) == text
print('Round-trip:', project_tokenizer.decode(ids)); print('IDs:', ids)

Round-trip: unseen 🧪 café नमस्ते
IDs: [118, 111, 116, 102, 102, 111, 33, 241, 160, 168, 171, 33, 267, 33, 263, 257, 175, 257, 185, 258, 142, 257, 165, 258, 136]
